# 02 - EDA, cleaning, affordability context
Analyses use `resale_price` medians (robust to expensive genuine sales). `price_per_sqm` is **EDA only** (target leakage if used to predict price).

In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd, numpy as np
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))
from paths import *
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"]=110
import data_cleaning as dc, feature_engineering as fe, geography_mapping as gm
def save(name): plt.tight_layout(); plt.savefig(FIGURES/f"{name}.png", bbox_inches="tight")
hdb = pd.read_csv(HDB_RAW)
hdb = dc.clean_hdb(hdb)
events = pd.read_csv(ROOT/"data"/"policy_events.csv", parse_dates=["effective_date"])
hdb = fe.add_policy_period(fe.add_property_features(fe.add_time_features(hdb)), events)
hdb = dc.flag_unusual_hdb(hdb)
print(hdb.shape); dc.rules_table()

## Data-quality summary

In [ ]:
qa = pd.Series({"raw rows": len(pd.read_csv(HDB_RAW)), "cleaned rows": len(hdb), "exact duplicates (flagged)": hdb.is_exact_duplicate.sum(),
 "any flagged record": hdb.any_flag.sum(), "date range": f"{hdb.transaction_date.min():%Y-%m} to {hdb.transaction_date.max():%Y-%m}", "towns": hdb.town.nunique(),
 "unique blocks": hdb.property_id.nunique()}); print(qa.to_string())
print(hdb.groupby("transaction_year").size().to_string())

## A. Market over time
Vertical lines mark policy events from `data/policy_events.csv` (only rows marked `verified_in_repo=yes` come from the HDB annual report; the rest are recalled and must be verified before citing).

In [ ]:
m = hdb.groupby("transaction_date").agg(volume=("resale_price","size"), median_price=("resale_price","median"), mean_price=("resale_price","mean"))
fig, ax = plt.subplots(2,1, figsize=(10,7), sharex=True)
ax[0].plot(m.index, m.median_price/1e3, label="Median"); ax[0].plot(m.index, m.mean_price/1e3, label="Mean", alpha=.6)
ax[0].set_ylabel("Resale price (S$ '000)"); ax[0].set_title("Monthly resale price"); ax[0].legend()
ax[1].bar(m.index, m.volume, width=25); ax[1].set_ylabel("Transactions"); ax[1].set_title("Monthly transaction volume")
for a in ax:
    for _, e in events.iterrows(): a.axvline(e.effective_date, color="red" if e.verified_in_repo=="yes" else "grey", ls="--", lw=.8)
save("a_market_over_time"); plt.show()
yr = hdb.groupby("transaction_year").agg(volume=("resale_price","size"), median_price=("resale_price","median"), median_psm=("price_per_sqm","median")).round(0)
yr["median_price_yoy_%"] = (yr.median_price.pct_change()*100).round(1); yr.to_csv(TABLES/"yearly_summary.csv"); yr

Note: 2026 is a partial year and the latest month may be incomplete; compare it with care.

## B. Town differences

In [ ]:
town = hdb.groupby("town").agg(n=("resale_price","size"), median_price=("resale_price","median"), median_area=("floor_area_sqm","median"), median_psm=("price_per_sqm","median")).sort_values("median_price")
town.round(0).to_csv(TABLES/"town_summary.csv")
fig, ax = plt.subplots(1,2, figsize=(12,7), sharey=True)
ax[0].barh(town.index, town.median_price/1e3); ax[0].set_xlabel("Median price (S$ '000)")
ax[1].barh(town.index, town.median_psm); ax[1].set_xlabel("Median price per sqm (S$)")
fig.suptitle("Town median resale price vs price per sqm (2017-2026)"); save("b_town"); plt.show()
town.round(0).tail(8)

Towns can be expensive because flats are bigger *or* because land/location commands a premium per sqm. Comparing the two panels (and `median_area` in the table) separates these effects.

## C/D. Flat type and town x flat type

In [ ]:
ft = hdb.groupby("flat_type").agg(n=("resale_price","size"), median_price=("resale_price","median"), median_psm=("price_per_sqm","median")).round(0); display(ft)
t = hdb[hdb.flat_type.isin(["3 ROOM","4 ROOM","5 ROOM","EXECUTIVE"])].groupby(["transaction_year","flat_type"]).resale_price.median().unstack()
t.plot(figsize=(9,4), marker="o", title="Median resale price by flat type"); plt.ylabel("S$"); save("c_flat_type_trend"); plt.show()
tf = hdb.groupby(["town","flat_type"]).agg(transaction_count=("resale_price","size"), median_price=("resale_price","median"), median_price_per_sqm=("price_per_sqm","median")).reset_index()
tf["low_sample"] = tf.transaction_count < 30
tf.round(0).to_csv(TABLES/"town_flat_type_summary.csv", index=False)
print("Low-sample (n<30) combinations:", tf.low_sample.sum(), "of", len(tf))

## E/F. Lease, age and storey

In [ ]:
s = hdb.sample(30000, random_state=0)
fig, ax = plt.subplots(1,3, figsize=(15,4))
ax[0].scatter(s.remaining_lease_months/12, s.resale_price/1e3, s=3, alpha=.2); ax[0].set(xlabel="Remaining lease (years)", ylabel="Price (S$ '000)", title="Remaining lease vs price")
ax[1].scatter(s.remaining_lease_months/12, s.price_per_sqm, s=3, alpha=.2); ax[1].set(xlabel="Remaining lease (years)", ylabel="S$/sqm", title="Remaining lease vs price/sqm")
sm = hdb.groupby("storey_mid").agg(psm=("price_per_sqm","median"), n=("resale_price","size")); sm = sm[sm.n>=50]
ax[2].plot(sm.index, sm.psm, marker="o"); ax[2].set(xlabel="Storey (mid)", ylabel="Median S$/sqm", title="Storey vs price/sqm (n>=50)")
save("ef_lease_storey"); plt.show()
print(hdb[["remaining_lease_months","flat_age_at_transaction","storey_mid","price_per_sqm","resale_price"]].corr(method="spearman").round(2))

Lease and age are strongly collinear with each other (and with location/flat model), so these are descriptive associations only.

## G. Population (Census 2020, static context)

In [ ]:
full, pa = dc.clean_population(pd.read_csv(POP_RAW)); pop = fe.population_features(pa)
mapping = gm.build_mapping(hdb.town.unique(), pop.planning_area)
display(mapping[mapping.mapping_status!="exact"][["hdb_town","population_planning_area","mapping_status","mapping_note"]])
tp = town.join(mapping.set_index("hdb_town").population_planning_area).merge(pop, left_on="population_planning_area", right_on="planning_area", how="left")
print(tp[["median_price","population_total"]].corr(method="spearman").round(2).iloc[0,1], "= Spearman(median price, area population)")

Population size is context, not an affordability measure; no further population plots are made because they would not inform pricing.

## H. Household income & affordability

In [ ]:
import household_income as hi
inc_wide, inc_long = hi.clean_income(pd.read_csv(INCOME_RAW)); inc = hi.income_features(inc_wide)
inc.to_csv(TABLES/"income_features.csv", index=False)
# distribution of work income across employed households, areas ordered by estimated mean
order = inc.sort_values("estimated_mean_household_income").planning_area
sh = inc_long.pivot(index="planning_area", columns="income_band", values="households"); sh = sh.div(sh.sum(axis=1), axis=0).loc[order]
grp = pd.DataFrame({"<$5k": sh[hi.BAND_COLS[:5]].sum(axis=1), "$5k-10k": sh[hi.BAND_COLS[5:10]].sum(axis=1), "$10k-15k": sh[hi.BAND_COLS[10:15]].sum(axis=1), "$15k-20k": sh[hi.BAND_COLS[15:17]].sum(axis=1), "$20k+": sh[hi.BAND_COLS[17]]})
grp.plot.barh(stacked=True, figsize=(9,8), colormap="viridis_r", width=.85); plt.xlabel("Share of households with employed persons"); plt.title("Household work-income distribution by planning area (Census 2020)")
plt.legend(bbox_to_anchor=(1,1)); save("h_income_distribution"); plt.show()
print("Mean sensitivity to the $20k+ assumption (S$ at 25k/30k/40k), e.g. Bukit Timah:", inc.set_index("planning_area").loc["Bukit Timah", [c for c in inc.columns if c.startswith("estimated_mean") and "open" in c]].astype(float).round(0).to_dict())

**Income method.** Median = linear interpolation inside the band containing the 50th percentile (assumes incomes spread evenly within a band), computed over households with an employed person. Not estimated where the median lies in the open `$20,000+` band (set to NaN; lower bound kept). Mean = band midpoints with `$20,000+` assumed at S$30,000 (default; 25k/40k sensitivities are stored). Raw band counts are in `household_income_bands_long.csv` for audit.

In [ ]:
aff = pd.read_csv(TABLES/"affordability_town_all_years.csv"); aff20 = pd.read_csv(TABLES/"affordability_town_2020.csv")
a = aff.merge(aff20[["town","median_price","price_to_income_ratio"]], on="town", suffixes=("","_2020"))
a = a.sort_values("price_to_income_ratio_2020")
fig, ax = plt.subplots(figsize=(8,7)); y = np.arange(len(a))
ax.barh(y-.2, a.price_to_income_ratio_2020, .4, label="2020 prices vs 2020 income (like-for-like)"); ax.barh(y+.2, a.price_to_income_ratio, .4, alpha=.6, label="2017-2026 prices vs 2020 income (static income)")
ax.set_yticks(y); ax.set_yticklabels(a.town); ax.set_xlabel("Median resale price / estimated annual household work income (years)"); ax.set_title("Housing-price-to-income indicator by town (not mortgage affordability)"); ax.legend(); save("h_price_to_income"); plt.show()
print("Towns with no estimable median income (open band) are excluded:", sorted(set(hdb.town.unique()) - set(a.town)))
a[["town","median_price_2020","estimated_median_household_income","price_to_income_ratio_2020","price_to_income_ratio"]].round(2).tail(8)

In [ ]:
# Are expensive towns expensive because of bigger flats or higher price/sqm? (2020)
t20 = hdb[hdb.transaction_year==2020].groupby("town").agg(price=("resale_price","median"), area=("floor_area_sqm","median"), psm=("price_per_sqm","median"))
t20 = t20.join(aff20.set_index("town")[["estimated_median_household_income"]])
fig, ax = plt.subplots(1,2, figsize=(12,4.5))
for a_, col, lab in [(ax[0],"price","Median resale price (S$)"), (ax[1],"psm","Median price per sqm (S$)")]:
    d = t20.dropna(); a_.scatter(d.estimated_median_household_income, d[col])
    for tn, r in d.iterrows(): a_.annotate(tn.title(), (r.estimated_median_household_income, r[col]), fontsize=6)
    a_.set(xlabel="Estimated median household work income (S$/month)", ylabel=lab)
fig.suptitle("2020: HDB resale levels vs local household income"); save("h_price_vs_income"); plt.show()
print("Spearman(income, price) =", t20.dropna().corr(method="spearman").loc["estimated_median_household_income",["price","area","psm"]].round(2).to_dict())

In [ ]:
aft = pd.read_csv(TABLES/"affordability_town_flat_type_2020.csv"); aft = aft[(~aft.low_sample)&aft.flat_type.isin(["3 ROOM","4 ROOM","5 ROOM","EXECUTIVE"])]
pv = aft.pivot(index="town", columns="flat_type", values="price_to_income_ratio").sort_values("4 ROOM")
sns.heatmap(pv, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label":"price / annual income"}); plt.title("2020 price-to-income indicator by town x flat type (n>=30)"); save("h_price_to_income_flat_type"); plt.show()

Caveats: the income column covers **all** resident households in the planning area (private and HDB), not just buyers; it is work income only; the ratio ignores CPF, grants, loans and household size. `Kallang/Whampoa` is an approximate match; `Central Area` is excluded. Where ratios use 2017-2026 prices against 2020 income they are descriptive only.

## I. Accessibility

In [ ]:
acc_p, loc_p = PROCESSED/"hdb_accessibility_features.csv", PROCESSED/"hdb_property_locations.csv"
# Guard: only analyse when most blocks are geocoded, otherwise results would be a biased sample of blocks.
acc_ready = False
if acc_p.exists():
    loc = pd.read_csv(loc_p); acc = pd.read_csv(acc_p)
    acc_ready = loc.latitude.notna().mean() >= 0.9
    if not acc_ready: print(f"SKIPPED: only {loc.latitude.notna().mean():.1%} of blocks geocoded (need >=90%). Finish src/run_geocoding.py, then python src/run_pipeline.py.")
else:
    print("Accessibility features not built yet - run python src/run_geocoding.py then python src/run_pipeline.py")
if acc_ready:
    n = len(loc); print(f"unique blocks {n}; geocoded {loc.latitude.notna().sum()} ({loc.latitude.notna().mean():.1%}); without coordinates {loc.latitude.isna().sum()}; with accessibility features {len(acc)}")
    print(loc.status.value_counts().to_dict())
    hx = hdb.merge(acc, on="property_id", how="inner")
    bands = [0,500,1000,1500,np.inf]; labels = ["0-500m","500m-1km","1-1.5km",">1.5km"]
    hx["mrt_band"] = pd.cut(hx.distance_to_nearest_mrt_m, bands, labels=labels, right=False)
    print(hx.groupby("mrt_band").agg(n=("resale_price","size"), median_price=("resale_price","median"), median_psm=("price_per_sqm","median")).round(0))

In [ ]:
if acc_ready:
    fig, ax = plt.subplots(1,2, figsize=(12,4.5))
    sns.boxplot(data=hx, x="mrt_band", y="price_per_sqm", ax=ax[0], showfliers=False); ax[0].set_title("Price per sqm by distance to nearest MRT/LRT station")
    sns.boxplot(data=hx, x="mrt_band", y="resale_price", ax=ax[1], showfliers=False); ax[1].set_title("Resale price by distance band")
    save("i_mrt_bands"); plt.show()
    # like-for-like: OLS of log price on distance band with town, flat type, lease, area, storey, year controls
    import statsmodels.formula.api as smf
    d = hx.dropna(subset=["mrt_band","remaining_lease_months","storey_mid"]).copy()
    d["log_price"] = np.log(d.resale_price); d["lease_yrs"] = d.remaining_lease_months/12
    d["mrt_band"] = pd.Categorical(d.mrt_band.astype(str), categories=labels, ordered=False)
    m0 = smf.ols("log_price ~ C(mrt_band, Treatment('0-500m'))", d).fit()
    m1 = smf.ols("log_price ~ C(mrt_band, Treatment('0-500m')) + C(town) + C(flat_type) + lease_yrs + floor_area_sqm + storey_mid + C(transaction_year)", d).fit(cov_type="HC1")
    pick = lambda m: (np.exp(m.params.filter(like="mrt_band"))-1).mul(100).round(1).rename(lambda i: i.split("[T.")[1].rstrip("]"))
    print("Approx % price difference vs 0-500m band:"); print(pd.DataFrame({"raw": pick(m0), "adjusted (town, type, lease, area, storey, year)": pick(m1)}))
    print("Association only - not causal: nearby unobserved factors (CBD proximity, amenities, flat model) are not controlled.")

In [ ]:
if acc_ready:
    sm_ = hx.groupby(["town"]).apply(lambda g: g.groupby("mrt_band").resale_price.median()).unstack().round(0)
    print("Median price by town and MRT band (towns where band n is small are noisy):"); display(sm_)
    print(hx[["distance_to_nearest_bus_stop_m","bus_stop_within_500m","mrt_within_1000m"]].describe().round(1))

## Year consistency
Census 2020 variables are **STATIC 2020 CONTEXTUAL FEATURES**. For direct comparison use only 2020 transactions (`hdb_2020_enriched`).